# Лабораторная работа 2

## Постановка задачи:

Решить задачу классификации датасета MNIST используя MLP из scikitlearn и используя CNN (по типу LeNet) c пакетом PyTorch. Сравнить результаты по метрикам, сделать обоснованные выводы.

## 1. Подготовка данных и MLP из scikit-learn

In [1]:
import numpy as np
import torch
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report
from torchvision import datasets, transforms

transform = transforms.ToTensor()
train_data = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_data = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

X_train = train_data.data.numpy().reshape(-1, 28*28) / 255.0
y_train = train_data.targets.numpy()
X_test = test_data.data.numpy().reshape(-1, 28*28) / 255.0
y_test = test_data.targets.numpy()

mlp = MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=20, alpha=1e-4,
                    solver='adam', verbose=True, random_state=42,
                    learning_rate_init=0.001)
mlp.fit(X_train, y_train)

y_pred_mlp = mlp.predict(X_test)
print("MLP Accuracy:", accuracy_score(y_test, y_pred_mlp))
print(classification_report(y_test, y_pred_mlp))

100%|██████████| 9.91M/9.91M [00:08<00:00, 1.14MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 281kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 2.47MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.75MB/s]


Iteration 1, loss = 0.38502588
Iteration 2, loss = 0.14950580
Iteration 3, loss = 0.10273425
Iteration 4, loss = 0.07876612
Iteration 5, loss = 0.06287742
Iteration 6, loss = 0.05150452
Iteration 7, loss = 0.04168628
Iteration 8, loss = 0.03421156
Iteration 9, loss = 0.02852425
Iteration 10, loss = 0.02430944
Iteration 11, loss = 0.02026940
Iteration 12, loss = 0.01670841
Iteration 13, loss = 0.01354651
Iteration 14, loss = 0.01485205
Iteration 15, loss = 0.01153144
Iteration 16, loss = 0.01095233
Iteration 17, loss = 0.00810442
Iteration 18, loss = 0.00709180
Iteration 19, loss = 0.00439719
Iteration 20, loss = 0.00553457
MLP Accuracy: 0.9769
              precision    recall  f1-score   support

           0       0.98      0.99      0.98       980
           1       0.99      0.99      0.99      1135
           2       0.98      0.96      0.97      1032
           3       0.96      0.99      0.97      1010
           4       0.97      0.98      0.97       982
           5       0.98

c:\Users\Roman\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(


## 2. Сверточная нейросеть (CNN) на PyTorch

In [2]:
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=1000, shuffle=False)

class LeNet(nn.Module):
    def __init__(self):
        super(LeNet, self).__init__()
        self.conv1 = nn.Conv2d(1, 6, 5, padding=2)
        self.act1 = nn.ReLU()
        self.pool1 = nn.AvgPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.act2 = nn.ReLU()
        self.pool2 = nn.AvgPool2d(2, 2)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.act3 = nn.ReLU()
        self.fc2 = nn.Linear(120, 84)
        self.act4 = nn.ReLU()
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool1(self.act1(self.conv1(x)))
        x = self.pool2(self.act2(self.conv2(x)))
        x = x.view(-1, 16 * 5 * 5)
        x = self.act4(self.fc2(self.act3(self.fc1(x))))
        x = self.fc3(x)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = LeNet().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 5
for epoch in range(epochs):
    model.train()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'CNN Accuracy: {100 * correct / total:.2f}%')

CNN Accuracy: 98.72%
